# W1 · Inspección de datos: horas extraordinarias en el personal a contrata del SSMC

**Propósito:** cargar, integrar y preparar la nómina de personal a contrata del Servicio de Salud Metropolitano Central (Transparencia Activa, código AO006) para inspeccionar las dos líneas candidatas del informe.

**Fuente:** Portal de Transparencia, 24 archivos mensuales (septiembre 2024 – agosto 2026), descargados el [fecha].

**Integrantes:** Alessandro Lavezzi - José Saavedra



In [1]:
import pandas as pd
import numpy as np
import seaborn as sbn
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

## 1. Lectura e integración de los archivos

Se leen los 24 archivos mensuales y se guardan en el diccionario `TA`, con nombres del tipo `TA_26_08` (año y mes).

Decisiones de lectura:
- **Separador `;` y codificación latin-1**, porque los archivos no están en UTF-8.
- **Todas las columnas como texto (`dtype=str`)**, porque varias mezclan formatos (por ejemplo, "Grado EUS o jornada" tiene "15" y "44 Hrs.", y los montos incluyen "$"). Así las conversiones se hacen después, de forma explícita.
- **Columna final vacía:** cada línea del archivo termina en `;`, lo que genera una columna sin nombre (`Unnamed: 21`). Se verifica que esté vacía y luego se elimina.
- **Columna `periodo`:** guarda el año-mes del archivo de origen, para identificar cada fila después de unir los meses.

In [ ]:
#LECTURA DE DATOS
DATA_DIR = Path("..") / "Data"
TA={}
for f in sorted(DATA_DIR.glob("TransparenciaActiva_*.csv")):
    fecha = f.stem.split("_")[-1]                  # "2026-08"
    nombre = f"TA_{fecha[2:4]}_{fecha[5:7]}"       # "TA_26_08"
    df = pd.read_csv(f, sep=";", encoding="latin-1",dtype=str)
    vacia = df["Unnamed: 21"].isna().all()         # la ultima columna esta vacia
    df = df.drop(columns=["Unnamed: 21"])          # se elimina
    df["periodo"] = fecha                          # se agrega columna año-mes del archivo
        
    
    globals()[nombre] = df
    TA[nombre] = df
    print(nombre, df.shape, "| columna vacía eliminada:", vacia)
    

TA_24_09 (1781, 22) | columna vacía eliminada: True
TA_24_10 (1799, 22) | columna vacía eliminada: True
TA_24_11 (1798, 22) | columna vacía eliminada: True
TA_24_12 (1803, 22) | columna vacía eliminada: True
TA_25_01 (1825, 22) | columna vacía eliminada: True
TA_25_02 (1842, 22) | columna vacía eliminada: True
TA_25_03 (1845, 22) | columna vacía eliminada: True
TA_25_04 (1861, 22) | columna vacía eliminada: True
TA_25_05 (1874, 22) | columna vacía eliminada: True
TA_25_06 (1870, 22) | columna vacía eliminada: True
TA_25_07 (1872, 22) | columna vacía eliminada: True
TA_25_08 (1884, 22) | columna vacía eliminada: True
TA_25_09 (1878, 22) | columna vacía eliminada: True
TA_25_10 (1882, 22) | columna vacía eliminada: True
TA_25_11 (1919, 22) | columna vacía eliminada: True
TA_25_12 (1918, 22) | columna vacía eliminada: True
TA_26_01 (1907, 22) | columna vacía eliminada: True
TA_26_02 (1951, 22) | columna vacía eliminada: True
TA_26_03 (1908, 22) | columna vacía eliminada: True
TA_26_04 (19

### 1.1 Unión de los 24 meses

Se unen los 24 archivos en una sola tabla (`df`). Para comprobar que no se perdieron ni duplicaron filas, se compara el número de filas de la tabla unida con la suma de las filas de los archivos individuales.

In [9]:
df=pd.concat(TA.values(), ignore_index=True)
suma_filas = sum(len(t) for t in TA.values())
print("Filas de la tabla unida:", len(df))
print("Suma de filas de los 24 meses:", suma_filas)
print("Columnas:", df.shape[1])
df

Filas de la tabla unida: 44777
Suma de filas de los 24 meses: 44777
Columnas: 22


,Año,Mes,Estamento,Nombre completo,Cargo o función,Grado EUS o jornada,Calificación profesional o formación,Región,Asignaciones especiales del mes (inc. en rem. bruta),"Remuneración bruta del mes (incluye bonos e incentivos, asig. especiales, horas extras)",Remuneración líquida del mes,Rem. adicionales del mes (no inc. en rem. bruta),Remuneración Bonos incentivos del mes (inc. en rem. bruta),Derecho a horas extraordinarias,Montos y horas extraordinarias diurnas del mes(inc. en rem. bruta),Montos y horas extraordinarias nocturnas del mes(inc. en rem. bruta),Montos y horas extraordinarias festivas del mes (inc. en rem. bruta),Fecha de inicio dd/mm/aa,Fecha de término dd/mm/aa,Viáticos del mes (no inc. en rem. bruta),Observaciones,periodo
0,2024,Septiembre,Auxiliar,"ABAITUA PIZARRO, GABRIEL ANTONIO",Chofer,20,AUXILIAR,Región Metropolitana de Santiago,(01),$ 838.003,$ 716.877,$ 0,$ 0,Sí,"$ 87.086 : 24,00 hrs",No tiene,No tiene,01/01/2024,31/12/2024,No informa,DIRECCION DE SERVICIO - REMUNERACION +BONO MEN...,2024-09
1,2024,Septiembre,Técnico,"ABARCA MATURANA, ARMANDO IVAN",ATENCION CLINICA,22,TECNICO NIVEL SUPERIOR ENFERMERIA,Región Metropolitana de Santiago,(01),$ 779.837,$ 564.066,$ 0,$ 0,No,No tiene,No tiene,No tiene,01/09/2024,30/09/2024,No informa,CENTRO METROPOLITANO DE ATENCION PREHOSPITALAR...,2024-09
2,2024,Septiembre,Profesional,"ABARCA MUÑOZ, MARITZA DENISSE",APOYO ADMINISTRATIVO,9,ABOGADO (A),Región Metropolitana de Santiago,(01),$ 2.404.046,$ 1.869.045,$ 0,$ 0,No,No tiene,No tiene,No tiene,01/01/2024,31/12/2024,No informa,CENTRO METROPOLITANO DE ATENCION PREHOSPITALAR...,2024-09
3,2024,Septiembre,Auxiliar,"ABARCA ROJAS, MANUEL EDUARDO",CONDUCTOR,24,AUXILIAR,Región Metropolitana de Santiago,(205)(203)(204)(186),$ 845.475,$ 655.884,$ 0,$ 94.920,No,No tiene,No tiene,No tiene,01/01/2024,31/12/2024,No informa,CENTRO METROPOLITANO DE ATENCION PREHOSPITALAR...,2024-09
4,2024,Septiembre,Técnico,"ABARCA RUBIO, FERNANDA DANIELA",ATENCION CLINICA,22,TECNICO NIVEL SUPERIOR ENFERMERIA,Región Metropolitana de Santiago,(202)(204)(186),$ 1.040.110,$ 503.890,$ 0,$ 112.666,No,No tiene,No tiene,No tiene,01/01/2024,31/12/2024,No informa,CENTRO METROPOLITANO DE ATENCION PREHOSPITALAR...,2024-09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44772,2026,Agosto,Auxiliar,"ZUÑIGA FLORES, MARCELO PAUL",CONDUCTOR,21,TECNICO NIVEL SUPERIOR ENFERMERIA,Región Metropolitana de Santiago,(205)(202)(204)(186),$ 1.587.591,$ 1.109.259,$ 0,$ 0,Sí,"$ 150.595 : 40,00 hrs","$ 189.749 : 42,00 hrs",No tiene,01/01/2026,31/12/2026,No informa,REMUNERACION AGO.2026,2026-08
44773,2026,Agosto,Profesional,"ZUÑIGA JARAMILLO, JONATHAN ROLANDO",Apoyo Administrativo,13,PSICOLOGO (A),Región Metropolitana de Santiago,(186),$ 1.908.175,$ 1.537.874,$ 0,$ 0,No,No tiene,No tiene,No tiene,01/01/2026,31/12/2026,No informa,Sin observaciones,2026-08
44774,2026,Agosto,Técnico,"ZUÑIGA OROZCO, ROMINA JAZMIN",ATENCION CLINICA,22,TECNICO NIVEL SUPERIOR ENFERMERIA,Región Metropolitana de Santiago,(202)(204)(186),$ 1.092.182,$ 824.661,$ 0,$ 0,No,No tiene,No tiene,No tiene,01/01/2026,31/12/2026,No informa,REMUNERACION AGO.2026,2026-08
44775,2026,Agosto,Técnico,"ZUÑIGA RIQUELME, CLAUDIO ALBERTO",ATENCION CLINICA,22,AUXILIAR DE ENFERMERIA,Región Metropolitana de Santiago,(202)(204)(186),$ 1.316.038,$ 858.615,$ 0,$ 0,Sí,"$ 52.347 : 15,00 hrs","$ 180.073 : 43,00 hrs",No tiene,01/01/2026,31/12/2026,No informa,REMUNERACION AGO.2026,2026-08


### Resultado de la sección 1

- Se cargaron los 24 archivos mensuales (septiembre 2024 – agosto 2026).
- La columna final sin nombre estaba vacía en los 24 archivos y se eliminó.
- La tabla unida tiene **44.777 filas**, igual a la suma de las filas de los 24 archivos, por lo que la unión no perdió ni duplicó registros.
- La tabla tiene 22 columnas: las 21 originales con información más `periodo`.
- El número de registros por mes varía entre 1.781 (septiembre 2024) y 1.951 (febrero 2026).

**Qué no permite concluir todavía:** cada fila es un registro del archivo, no necesariamente una persona distinta. Si una persona tiene más de un contrato, aparece en más de una fila. Esto se revisa en la sección 2.

## 2. Preparación de los datos

### 2.1 Conversión de montos de remuneración

Los montos vienen como texto, con signo peso y punto como separador de miles (ej. "$ 1.512.586"). Para poder sumarlos y compararlos se convierten a número entero, en pesos chilenos.

Se convierten cuatro columnas: remuneración bruta, remuneración líquida, remuneraciones adicionales y bonos. Se cuenta cuántos valores no se pudieron convertir y se muestra cuáles son, para revisarlos antes de seguir.

In [11]:
#CONVERSIÓN DE MONTOS
COLS_MONTO = {
    "Remuneración bruta del mes (incluye bonos e incentivos, asig. especiales, horas extras)": "rem_bruta",
    "Remuneración líquida del mes": "rem_liquida",
    "Rem. adicionales del mes (no inc. en rem. bruta)": "rem_adicionales",
    "Remuneración Bonos incentivos del mes (inc. en rem. bruta)": "bonos",
}

In [12]:
def a_monto(serie):
    limpio = (
        serie.str.replace("$", "", regex=False)   # quita el signo peso
             .str.replace(".", "", regex=False)   # quita el punto de miles
             .str.strip()                         # quita espacios al inicio y al final
    )
    return pd.to_numeric(limpio, errors="coerce")

In [13]:
control_montos = []

for original, nuevo in COLS_MONTO.items():
    df[nuevo] = a_monto(df[original])

    no_convertibles = df[nuevo].isna() & df[original].notna()
    control_montos.append({
        "columna": nuevo,
        "no_convertibles": no_convertibles.sum(),
        "valores": df.loc[no_convertibles, original].value_counts().to_dict(),
        "meses": df.loc[no_convertibles, "periodo"].value_counts().to_dict(),
    })

pd.DataFrame(control_montos)

,columna,no_convertibles,valores,meses
0,rem_bruta,0,{},{}
1,rem_liquida,0,{},{}
2,rem_adicionales,1825,{'-': 1825},{'2025-01': 1825}
3,bonos,3065,{'-': 3065},"{'2025-05': 1507, '2026-07': 1479, '2024-12': ..."


In [14]:
control_montos[3]["meses"]

{'2025-05': 1507, '2026-07': 1479, '2024-12': 70, '2025-11': 9}

**Resultado de la conversión de montos**

- La remuneración bruta y la remuneración líquida se convirtieron sin errores en los 44.777 registros.
- En `rem_adicionales`, 1.825 valores contienen "-" en lugar de un monto. Todos corresponden a enero de 2025, que tiene 1.825 registros: en ese mes la columna completa viene con "-".
- En `bonos`, 3.065 valores contienen "-", concentrados en algunos meses: [completar con los meses de la salida].

**Decisión:** los valores "-" se dejan como vacíos (`NaN`) y no se reemplazan por 0, porque la fuente no indica si significan "sin monto" o "no informado". Estas dos columnas no se usan directamente en las candidatas; la Candidata B utiliza la remuneración bruta, que incluye los bonos y no presenta valores no convertibles.

### 2.2 Separación de monto y horas en las columnas de horas extraordinarias

Las tres columnas de horas extraordinarias (diurnas, nocturnas y festivas) combinan en un mismo texto el monto pagado y el número de horas, con el formato "$ 154.547 : 40,00 hrs". Cuando no hay horas de ese tipo, dicen "No tiene".

Cada columna se separa en dos variables numéricas:
- `monto_<tipo>`: monto pagado, en pesos.
- `horas_<tipo>`: número de horas (la coma decimal se convierte a punto).

**Decisión:** "No tiene" se codifica como 0, porque indica que no se pagaron horas extraordinarias de ese tipo en el mes, no que el dato falte.

Se cuenta cuántos valores tienen cada formato y cuántos no calzan con ninguno.

In [18]:
#SEPARACIÓN DE HORAS EXTRAORDINARIAS
COLS_HHEE = {
    "Montos y horas extraordinarias diurnas del mes(inc. en rem. bruta)": "diurnas",
    "Montos y horas extraordinarias nocturnas del mes(inc. en rem. bruta)": "nocturnas",
    "Montos y horas extraordinarias festivas del mes (inc. en rem. bruta)": "festivas",
}

control_hhee = []

for original, tipo in COLS_HHEE.items():
    texto = df[original].str.strip()
    no_tiene = texto.eq("No tiene")

    partes = texto.str.split(":", n=1, expand=True).reindex(columns=[0, 1]).astype("string")
    monto = a_monto(partes[0])                   # misma función del paso 2.1
    horas = pd.to_numeric(
        partes[1].str.replace("hrs", "", regex=False)
                 .str.replace(".", "", regex=False)
                 .str.replace(",", ".", regex=False)
                 .str.strip(),
        errors="coerce")

    df[f"monto_{tipo}"] = monto.where(~no_tiene, 0)
    df[f"horas_{tipo}"] = horas.where(~no_tiene, 0)

    control_hhee.append({
        "tipo": tipo,
        "con_monto_y_horas": (monto.notna() & horas.notna()).sum(),
        "no_tiene": no_tiene.sum(),
        "vacios": texto.isna().sum(),
        "otro_formato": ((monto.isna() | horas.isna()) & ~no_tiene & texto.notna()).sum(),
    })

pd.DataFrame(control_hhee)

,tipo,con_monto_y_horas,no_tiene,vacios,otro_formato
0,diurnas,10956,33821,0,0
1,nocturnas,8226,36551,0,0
2,festivas,0,44777,0,0


**Resultado de la separación de horas extraordinarias**

- En los tres tipos de horas extraordinarias, todos los valores tenían uno de los dos formatos esperados ("monto : horas hrs" o "No tiene"). No hubo valores vacíos ni con otro formato, por lo que la conversión fue completa.
- Horas diurnas: 10.956 registros (24,5%) tienen horas pagadas.
- Horas nocturnas: 8.226 registros (18,4%) tienen horas pagadas.
- Horas festivas: ningún registro tiene horas pagadas; la columna indica "No tiene" en los 44.777 registros de los 24 meses.

**Qué no permite concluir:** los porcentajes se refieren a registros, no a personas, porque una persona puede tener más de un contrato. Tampoco permite saber si las horas festivas no se realizan, si se registran como nocturnas o si simplemente no se informan en esta columna.



### 2.3 Conversión de fechas

Las columnas "Fecha de inicio" y "Fecha de término" indican la vigencia de cada contrato. Vienen como texto, y aunque su encabezado dice "dd/mm/aa", los valores traen el año con cuatro dígitos (ej. 01/01/2026). Se convierten a formato de fecha usando día/mes/año completo, y se cuenta cuántas fechas no se pudieron convertir.

In [19]:
#CONVERSIÓN DE FECHAS
COLS_FECHA = {
    "Fecha de inicio dd/mm/aa": "fecha_inicio",
    "Fecha de término dd/mm/aa": "fecha_termino",
}

for original, nuevo in COLS_FECHA.items():
    df[nuevo] = pd.to_datetime(df[original], format="%d/%m/%Y", errors="coerce")

    vacias = df[original].isna().sum()
    no_convertibles = (df[nuevo].isna() & df[original].notna()).sum()
    print(f"{nuevo}: vacías = {vacias}, no convertibles = {no_convertibles}")

fecha_inicio: vacías = 0, no convertibles = 0
fecha_termino: vacías = 0, no convertibles = 0


**Resultado de la conversión de fechas**

- Las fechas de inicio y término se convirtieron sin errores en los 44.777 registros: ninguna estaba vacía y todas tenían el formato día/mes/año con el año en cuatro dígitos.
- Se confirma que el encabezado del archivo ("dd/mm/aa") no describe correctamente el formato de los valores, que usan el año completo.

## 3. Unidad de observación

Se determina si cada fila corresponde a una persona o a un contrato. Para ello, se cuenta cuántas veces aparece cada nombre dentro de un mismo mes. Si un nombre aparece más de una vez en el mismo mes, esa persona tiene más de un registro (por ejemplo, dos contratos con jornadas distintas).

Se usa el nombre completo porque la nómina no incluye RUT. Las salidas muestran solo conteos, no nombres.

In [20]:
#UNIDAD DE OBSERVACIÓN
por_mes = df.groupby("periodo").agg(
    filas=("Nombre completo", "size"),
    personas_distintas=("Nombre completo", "nunique"),
)
por_mes["filas_extra"] = por_mes["filas"] - por_mes["personas_distintas"]

repeticiones = df.groupby(["periodo", "Nombre completo"]).size()
por_mes["personas_con_mas_de_un_registro"] = (repeticiones > 1).groupby("periodo").sum()

por_mes

,filas,personas_distintas,filas_extra,personas_con_mas_de_un_registro
periodo,,,,
2024-09,1781,1717,64,60
2024-10,1799,1733,66,62
2024-11,1798,1734,64,61
2024-12,1803,1739,64,60
2025-01,1825,1756,69,63
2025-02,1842,1776,66,60
2025-03,1845,1783,62,57
2025-04,1861,1798,63,59
2025-05,1874,1804,70,66


In [21]:
print("Filas totales:", len(df))
print("Personas distintas en los 24 meses:", df["Nombre completo"].nunique())
print("Meses con al menos una persona repetida:", (por_mes["filas_extra"] > 0).sum(), "de", len(por_mes))
print("Máximo de registros de una misma persona en un mes:", repeticiones.max())

Filas totales: 44777
Personas distintas en los 24 meses: 2430
Meses con al menos una persona repetida: 24 de 24
Máximo de registros de una misma persona en un mes: 6


**Resultado de la unidad de observación**

- En los 24 meses hay personas con más de un registro en el mismo mes: entre 57 y 68 personas por mes, que generan entre 62 y 76 filas adicionales (entre 3,4% y 4,0% de las filas del mes). Una misma persona llegó a tener hasta 6 registros en un mes.
- Por lo tanto, **la unidad de observación es el registro de contrato o nombramiento en un mes, no la persona**.
- En los 24 meses aparecen 2.430 personas distintas, más que las que hay en cualquier mes individual (entre 1.717 y 1.880), lo que indica rotación de personal durante el período.

**Consecuencias para el análisis:**
- Candidata A: el límite legal de 40 horas diurnas es por funcionario al mes, por lo que las horas deben sumarse por persona y mes antes de compararlas con el límite.
- Candidata B: para medir persistencia, los registros deben agruparse por persona y mes.

**Limitación:** sin RUT, el nombre no es un identificador perfecto. Dos personas con el mismo nombre se contarían como una, y una persona con el nombre escrito de forma distinta en algún mes se contaría como dos. Esta fuente no permite verificar ninguno de los dos casos.

## 4. Primeras observaciones para la Candidata A

**Pregunta:** ¿qué proporción del personal a contrata registra pagos por más de 40 horas extraordinarias diurnas en un mes, y en qué estamentos se concentran las horas nocturnas atípicamente altas?

### 4.1 Horas por persona y mes

El límite de la Ley 19.104 es de 40 horas extraordinarias diurnas **por funcionario al mes**. Como una persona puede tener más de un registro en el mismo mes (sección 3), primero se suman sus horas de todos sus registros. Cada fila de la nueva tabla es una **persona-mes**.

Para asignar un estamento a cada persona-mes se usa el estamento de su primer registro del mes.

In [22]:
#HORAS POR PERSONA Y MES
pm = (
    df.groupby(["periodo", "Nombre completo"])
      .agg(
          Estamento=("Estamento", "first"),
          horas_diurnas=("horas_diurnas", "sum"),
          horas_nocturnas=("horas_nocturnas", "sum"),
      )
      .reset_index()
)

pm["sobre_40"] = pm["horas_diurnas"] > 40

print("Persona-mes totales:", len(pm))
print("Persona-mes con horas diurnas:", (pm["horas_diurnas"] > 0).sum())
print("Persona-mes con más de 40 horas diurnas:", pm["sobre_40"].sum())
print("Persona-mes con horas nocturnas:", (pm["horas_nocturnas"] > 0).sum())

Persona-mes totales: 43120
Persona-mes con horas diurnas: 10946
Persona-mes con más de 40 horas diurnas: 182
Persona-mes con horas nocturnas: 8222


### Control: cambios de estamento entre meses consecutivos

Se compara el estamento de cada persona entre un mes y el siguiente. Un cambio de estamento es poco frecuente, por lo que un número alto de cambios en un mes puede indicar un problema de registro en ese archivo.

In [25]:
#CAMBIOS DE ESTAMENTO ENTRE MESES
est = pm[["periodo", "Nombre completo", "Estamento"]].sort_values(["Nombre completo", "periodo"])
est["estamento_mes_anterior"] = est.groupby("Nombre completo")["Estamento"].shift(1)

cambios = est.dropna(subset=["estamento_mes_anterior"])
cambios = cambios[cambios["Estamento"] != cambios["estamento_mes_anterior"]]

cambios.groupby("periodo").size()

periodo
2024-12      1
2025-01      2
2025-02      1
2025-09      1
2025-11      1
2025-12    218
2026-01    213
2026-02      6
2026-08      1
dtype: int64

**Resultado del control de estamento**

- En diciembre de 2025, 218 personas cambian de estamento respecto del mes anterior, y en enero de 2026, 213 personas cambian de nuevo. En el resto de los meses hay entre 0 y 6 cambios.
- El patrón indica que esas personas cambian de estamento solo en diciembre de 2025 y vuelven al original en enero de 2026. Esto sugiere un error de registro en la columna de estamento del archivo de diciembre de 2025.

**Decisión:** diciembre de 2025 se excluye de los análisis por estamento (sección 4.2). Se mantiene en los análisis que no dependen del estamento, como los casos por mes (sección 4.3).

### 4.2 Resumen por estamento

Para cada estamento se calcula:
- cuántas persona-mes tienen horas diurnas y cuántas superan las 40 horas;
- la mediana y el máximo de horas diurnas, entre quienes tienen horas diurnas;
- la mediana, el percentil 95 y el máximo de horas nocturnas, entre quienes tienen horas nocturnas.

Se usa la mediana en vez del promedio porque es menos sensible a valores extremos. El percentil 95 indica el valor bajo el cual está el 95% de los casos, y sirve para ver qué tan altos son los casos atípicos.

El estamento médico se presenta por separado porque, según la normativa, el límite de 40 horas no le aplica de la misma forma (sección 1.2.2 del informe).

In [26]:
#RESUMEN POR ESTAMENTO
def resumen(g):
    diurnas = g.loc[g["horas_diurnas"] > 0, "horas_diurnas"]
    nocturnas = g.loc[g["horas_nocturnas"] > 0, "horas_nocturnas"]
    return pd.Series({
        "persona_mes": len(g),
        "con_diurnas": len(diurnas),
        "sobre_40_diurnas": g["sobre_40"].sum(),
        "mediana_diurnas": diurnas.median(),
        "max_diurnas": diurnas.max(),
        "con_nocturnas": len(nocturnas),
        "mediana_nocturnas": nocturnas.median(),
        "p95_nocturnas": nocturnas.quantile(0.95),
        "max_nocturnas": nocturnas.max(),
    })

pm_estamento = pm[pm["periodo"] != "2025-12"]      # se excluye dic-2025 por error en el estamento
tabla_A = pm_estamento.groupby("Estamento")[pm_estamento.columns].apply(resumen)
tabla_A

,persona_mes,con_diurnas,sobre_40_diurnas,mediana_diurnas,max_diurnas,con_nocturnas,mediana_nocturnas,p95_nocturnas,max_nocturnas
Estamento,,,,,,,,,
Administrativo,5642.000,2679.000,40.000,21.000,80.000,1559.000,20.000,104.000,205.000
Auxiliar,2863.000,1735.000,49.000,30.000,80.000,1771.000,56.000,192.500,418.000
"Médicos cirujanos, farmacéuticos, químico-farmacéuticos, bioquímicos, cirujano-dentistas",7936.000,293.000,0.000,20.000,40.000,266.000,57.000,105.000,207.000
Profesional,16805.000,3678.000,48.000,18.000,160.000,2528.000,15.000,110.000,279.000
Técnico,8026.000,2108.000,39.000,20.000,80.000,1755.000,32.000,120.000,321.000


In [29]:
#CIFRAS PARA LA INTERPRETACIÓN DE LA TABLA A
totales = tabla_A[["persona_mes", "con_diurnas", "sobre_40_diurnas", "con_nocturnas"]].sum()
print("Totales sin diciembre 2025:")
print(totales.astype(int))

print("\n% de persona-mes con horas diurnas sobre 40, entre quienes tienen horas diurnas:")
print("Total:", round(totales["sobre_40_diurnas"] / totales["con_diurnas"] * 100, 1), "%")

pct_sobre_40 = (tabla_A["sobre_40_diurnas"] / tabla_A["con_diurnas"] * 100).round(1)
print(pct_sobre_40.sort_values(ascending=False))

print("\n% de persona-mes con horas diurnas, por estamento:")
pct_con_diurnas = (tabla_A["con_diurnas"] / tabla_A["persona_mes"] * 100).round(1)
print(pct_con_diurnas.sort_values(ascending=False))

max_noct = tabla_A["max_nocturnas"].max()
print("\nMáximo de horas nocturnas en un mes:", max_noct)
print("Equivale a horas nocturnas por día (mes de 30 días):", round(max_noct / 30, 1))

Totales sin diciembre 2025:
persona_mes         41272
con_diurnas         10493
sobre_40_diurnas      176
con_nocturnas        7879
dtype: int64

% de persona-mes con horas diurnas sobre 40, entre quienes tienen horas diurnas:
Total: 1.7 %
Estamento
Auxiliar                                                                                   2.800
Técnico                                                                                    1.900
Administrativo                                                                             1.500
Profesional                                                                                1.300
Médicos cirujanos, farmacéuticos, químico-farmacéuticos, bioquímicos, cirujano-dentistas   0.000
dtype: float64

% de persona-mes con horas diurnas, por estamento:
Estamento
Auxiliar                                                                                   60.600
Administrativo                                                                            

**Resultado del resumen por estamento** (se excluye diciembre de 2025 por el error de registro en la columna de estamento detectado en el control anterior)

- En los 23 meses considerados hay 41.272 persona-mes, de las cuales 10.493 tienen horas diurnas pagadas y 176 superan las 40 horas diurnas (1,7% de las persona-mes con horas diurnas).
- El estamento médico no presenta casos sobre 40 horas diurnas (máximo: 40).
- En proporción, los auxiliares concentran más casos sobre el límite (2,8% de sus persona-mes con horas diurnas), seguidos de técnicos (1,9%), administrativos (1,5%) y profesionales (1,3%).
- Las horas diurnas son más frecuentes entre auxiliares (60,6% de sus persona-mes) y administrativos (47,5%), y poco frecuentes en el estamento médico (3,7%).
- Los auxiliares registran las horas nocturnas más altas: mediana de 56 horas, percentil 95 de 192,5 y máximo de 418 horas en un mes.

**Qué no permite concluir:** estos datos corresponden a horas pagadas, no a horas trabajadas. Un valor de 418 horas nocturnas equivale a unas 14 horas nocturnas diarias durante un mes de 30 días, lo que sugiere que algunos pagos podrían incluir horas de meses anteriores; esta fuente no permite verificarlo. Tampoco permite saber si los casos sobre 40 horas cuentan con una excepción autorizada.

### 4.3 Casos sobre 40 horas diurnas por mes

Se cuenta cuántas persona-mes superan las 40 horas diurnas en cada mes, para ver si los casos se distribuyen de forma pareja o se concentran en algún período.

In [28]:
#CASOS SOBRE 40 HORAS POR MES
sobre_40_por_mes = pm.groupby("periodo")["sobre_40"].sum()
sobre_40_por_mes

periodo
2024-09     2
2024-10     2
2024-11     2
2024-12     6
2025-01    99
2025-02     6
2025-03     4
2025-04     8
2025-05     5
2025-06     5
2025-07     5
2025-08     4
2025-09     5
2025-10     5
2025-11     0
2025-12     6
2026-01     2
2026-02     4
2026-03     4
2026-04     0
2026-05     2
2026-06     0
2026-07     3
2026-08     3
Name: sobre_40, dtype: Int64

**Resultado de casos sobre 40 horas por mes**

- Enero de 2025 concentra 99 de los 182 casos del período (54%).
- Los otros 23 meses suman 83 casos, con entre 0 y 8 casos por mes.

**Qué no permite concluir:** enero de 2025 también es el único mes en que la columna de remuneraciones adicionales viene completa con "-" (sección 2.1). El pico podría reflejar una necesidad operativa real de ese mes o una forma distinta de registrar el pago en ese archivo (por ejemplo, horas de meses anteriores pagadas en enero). Esta fuente no permite distinguir entre ambas explicaciones.